<a href="https://colab.research.google.com/github/maryamazzam111-spec/projects-power-BI/blob/main/1st_confounded_factors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install psmpy -q

In [2]:
# 2. استدعاء المكتبات الأساسية
import pandas as pd
import numpy as np
from psmpy import PsmPy

In [3]:
# 3. إنشاء داتا افتراضية تشبه واقع دراسة الحالة بتاعتنا بالضبط (500 شخص)
np.random.seed(42)
n = 500

In [4]:
# العوامل المربكة (Confounders)
age = np.random.randint(21, 30, size=n)
location = np.random.choice([0, 1], size=n, p=[0.4, 0.6]) # 0=ريف، 1=حضر
motivation = np.random.normal(loc=50, scale=10, size=n) # مؤشر الدافعية

# احتمال دخول الكورس (بيعتمد على الدافعية والمكان)
prob_course = 1 / (1 + np.exp(-(-2 + 0.05*motivation + 0.8*location)))
got_course = np.random.binomial(1, prob_course)

# التوظيف (بيعتمد بشكل قوي على الدافعية والمكان، وأثر بسيط جداً للكورس نفسه)
prob_employed = 1 / (1 + np.exp(-(-3 + 0.08*motivation + 1.2*location + 0.3*got_course)))
employed = np.random.binomial(1, prob_employed)

In [5]:

# تجهيز الجدول
df = pd.DataFrame({
    'age': age,
    'location': location,
    'motivation': motivation,
    'got_course': got_course,
    'employed': employed
})

# عرض أول 5 صفوف من الداتا
df.head()

,age,location,motivation,got_course,employed
0,27,1,40.934213,1,1
1,24,1,58.444424,1,1
2,28,0,49.011371,1,1
3,25,1,72.979590,1,1
4,27,0,61.861782,1,1


In [6]:
# 1. حساب نسبة التوظيف للناس اللي أخدت الكورس (got_course = 1)
raw_treated = df[df['got_course'] == 1]['employed'].mean()

# 2. حساب نسبة التوظيف للناس اللي مأخدتش الكورس (got_course = 0)
raw_control = df[df['got_course'] == 0]['employed'].mean()

# 3. الفرق الظاهري الخادع
raw_effect = raw_treated - raw_control

print(f"نسبة توظيف اللي أخدوا الكورس (ظاهرياً): {raw_treated*100:.1f}%")
print(f"نسبة توظيف اللي مأخدوش الكورس (ظاهرياً): {raw_control*100:.1f}%")
print(f"الفرق الظاهري الخادع (Raw Effect): {raw_effect*100:.1f}%")


نسبة توظيف اللي أخدوا الكورس (ظاهرياً): 87.0%
نسبة توظيف اللي مأخدوش الكورس (ظاهرياً): 74.8%
الفرق الظاهري الخادع (Raw Effect): 12.2%


In [10]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. الموديل الظاهري (الخام من غير عزل العوامل) - اللي بيطلع الرقم المضلل
raw_model = smf.logit("employed ~ got_course", data=df).fit(disp=0)
print("--- 1. النتيجة الخام الظاهرية ---")
print(f"أثر الكورس ظاهرياً: {raw_model.params['got_course']:.4f}\n")

# 2. الموديل الصحيح السببي (العادل) - بنثبت فيه العوامل المربكة (الدافعية والمكان والعمر)
causal_model = smf.logit("employed ~ got_course + motivation + location + age", data=df).fit(disp=0)

print("--- 2. النتيجة الحقيقية بعد عزل المتغيرات المربكة (Causal Effect) ---")
print(causal_model.summary().tables[1])

--- 1. النتيجة الخام الظاهرية ---
أثر الكورس ظاهرياً: 0.8134

--- 2. النتيجة الحقيقية بعد عزل المتغيرات المربكة (Causal Effect) ---
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.2746      1.392      0.197      0.844      -2.453       3.002
got_course     0.2593      0.281      0.921      0.357      -0.292       0.811
motivation     0.0562      0.013      4.174      0.000       0.030       0.083
location       1.2747      0.269      4.733      0.000       0.747       1.803
age           -0.0891      0.051     -1.752      0.080      -0.189       0.011
